# 01 - טעינת נתונים וניקוי

זהו השלב הראשון בצינור העיבוד (pipeline) של הפרויקט. הוא קורא את קובץ ה-GTFS הגולמי של ישראל (ייצוא לוחות הזמנים הארצי של התחבורה הציבורית), ממיר את טבלאות הטקסט למסגרות נתונים (data frames) בעלות טיפוסים מוגדרים, מחיל מערך מצומצם ומפורש של מסנני שפיות, ומצרף לכל תחנה שתי תוויות גאוגרפיות גסות: **אזור** (region: North / Center / Jerusalem / South) ו**מטרופולין** (metro: Tel Aviv / Haifa / Jerusalem / Beer Sheva / Periphery). כל שאר שלבי הפרויקט - בניית הגרף, מדדי מרכזיות, עמידות - מתחילים מהטבלאות המנוקות הנכתבות כאן, ולכן מחברת זו מפיקה גם דוח ניקוי קריא-מכונה המתעד במדויק כמה שורות הסיר כל מסנן.

הרעיון המנחה הוא שמחקר רשתות אמין בדיוק כמידת האמינות של טבלת הצמתים שלו. אם לתחנה יש קואורדינטה שבורה, או אם אותו `stop_id` מופיע פעמיים, הגרף מקבל בשקט צומת רפאים וכל ערכי המרכזיות שיחושבו לאחר מכן מזוהמים. לפיכך כל מסנן כאן נכתב כשלב עצמאי, נספר, ומדווח.

**שאלת המחקר שהשלב הזה משרת:** *מהי בדיוק קבוצת הצמתים של רשת התחבורה הציבורית בישראל, ומה השלכנו כדי להגיע אליה?*

### קלט

- `israel-public-transportation/stops.txt` - שורה אחת לכל תחנה / תחנת רכבת פיזית (מזהה, שם, קו רוחב, קו אורך, סוג מיקום, תחנת אב).
- `israel-public-transportation/routes.txt` - שורה אחת לכל קו, עם `route_type` (3 = אוטובוס, 2 = רכבת, ...) ועם `agency_id` של המפעיל.
- `israel-public-transportation/trips.txt` - שורה אחת לכל נסיעה מתוזמנת של כלי רכב; זוהי טבלת נפח השירות.
- `israel-public-transportation/agency.txt` - שורה אחת לכל מפעיל.

כל ארבעת הקבצים מנוהלים בתוך המאגר (repository). מחברת זו **אינה** נוגעת ב-`stop_times.txt` (816 MB, אינו מנוהל ב-git) - קובץ זה נדרש רק ממחברת 02 והלאה.

### פלט (הכול תחת `outputs/nb/01_data_preparation/`)

- `tables/stops_clean.csv` - טבלת הצמתים המנוקה, בתוספת העמודות החדשות `region` ו-`metro`.
- `tables/routes_clean.csv`, `tables/trips_clean.csv`, `tables/agencies_clean.csv` - עותקי מעבר (pass-through) עם טיפוסים יציבים, כך שמחברות מאוחרות לא נדרשות לגזור מחדש את אפשרויות הפרסור.
- `tables/route_type_distribution.csv` - מספר הקווים והנסיעות המתוזמנות לפי אופן תחבורה (mode) של GTFS.
- `tables/agency_trip_counts.csv` - נסיעות מתוזמנות לכל מפעיל.
- `tables/data_cleaning_report.json` - מסלול הביקורת (audit trail): ספירות שורות לפני ואחרי כל מסנן, פילוחי region/metro, ובדיקות שלמות רפרנציאלית.
- `figures/*.png` - ארבעה איורים תיאוריים.

### תלויות

אין. זהו שלב 01; הוא קורא אך ורק את הקובץ הגולמי.

## אתחול סביבת העבודה

התא שלהלן זהה בכל מחברות הפרויקט. הוא מבצע שלושה דברים: (1) מגדיר את `_ensure`, המתקין באמצעות pip *רק* את החבילות החסרות בפועל, כך שהרצה חוזרת אינה עולה דבר; (2) מאתר את שורש המאגר על ידי טיפוס כלפי מעלה מהתיקייה הנוכחית בחיפוש אחר התיקייה `israel-public-transportation`, ומשכפל (clone) את המאגר אם אנו על מכונת Google Colab חדשה; (3) יוצר את התיקייה המשותפת `outputs/nb`. לאחר תא זה, `REPO`, `DATA` ו-`OUT` זמינים ותיקיית העבודה היא שורש המאגר, כך שניתן לכתוב כל נתיב במחברת ביחס לעוגן ידוע.

In [ ]:
# --- Environment bootstrap (safe to re-run, works locally and on Google Colab) ---
import os, sys, subprocess
from pathlib import Path

def _ensure(*pkgs):
    """Install only the packages that are actually missing."""
    import importlib.util
    alias = {"scikit-learn": "sklearn", "python-louvain": "community",
             "python-bidi": "bidi", "node2vec": "node2vec"}
    missing = [p for p in pkgs
               if importlib.util.find_spec(alias.get(p, p.replace("-", "_"))) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

def find_repo_root():
    """Find the repo locally; on Colab, clone it."""
    here = Path(os.getcwd()).resolve()
    for cand in [here, *here.parents]:
        if (cand / "israel-public-transportation").is_dir():
            return cand
    target = Path("/content/israel-transit-network-resilience")
    if not target.exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/seanfourman/israel-transit-network-resilience.git",
                        str(target)], check=True)
    return target

REPO = find_repo_root()
os.chdir(REPO)
DATA = REPO / "israel-public-transportation"
OUT = REPO / "outputs" / "nb"
OUT.mkdir(parents=True, exist_ok=True)
print("Repo root:", REPO)

## ספריות, תיקיית השלב וקבועים ניתנים לכיוונון

מחברת זו מחזיקה בתיקייה `outputs/nb/01_data_preparation/`, המחולקת ל-`tables/` ול-`figures/`. שום דבר אינו נכתב לעולם אל `outputs/tables`, `outputs/figures` או `outputs/rail`, המכילות את התוצאות המצוטטות כבר בדוח הכתוב.

שתי קבוצות של קבועים רוכזו כאן כדי שהקורא יוכל לראות ולשנות כל פרמטר ניתן לכיוונון במקום אחד:

- **`LAT_MIN/LAT_MAX/LON_MIN/LON_MAX`** - תיבת התיחום (bounding box) שקואורדינטה חייבת ליפול בתוכה כדי להתקבל כתחנה ישראלית אמיתית. התיבה רחבה במכוון (אילת נמצאת סביב 29.55 צפון, מטולה סביב 33.28 צפון), כך שהיא לוכדת רק ערכים שבורים בעליל כגון מציין מקום `0,0`, ולא תחנות לגיטימיות בקצה המדינה.
- **`SAVE_TRIPS_CSV`** - הקובץ `trips.txt` שוקל כ-28 MB וכולל בקירוב 1.2 מיליון שורות; כתיבתו מחדש כ-CSV היא הפעולה האיטית ביותר במחברת זו (עשרות שניות ו-28 MB של שטח דיסק). יש להגדיר אותו כ-`False` אם מעניינות אתכם רק טבלת התחנות והדוח.

In [ ]:
# Third-party libraries used below. _ensure only installs what is missing.
_ensure("pandas", "numpy", "matplotlib")

import json
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --- This notebook's own output folder (stage 01) ---
STAGE = OUT / "01_data_preparation"
TABLES = STAGE / "tables"
FIGURES = STAGE / "figures"
TABLES.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

# --- Tunable constants ---
LAT_MIN, LAT_MAX = 29.0, 34.0   # generous bounding box around Israel
LON_MIN, LON_MAX = 34.0, 36.0
SAVE_TRIPS_CSV = True           # writing trips_clean.csv costs ~28 MB and tens of seconds

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

print("GTFS folder :", DATA)
print("Stage folder:", STAGE)

## הצגה נכונה של תוויות בעברית

כל שמות התחנות ושמות המפעילים בקובץ הם בעברית, ואחד האיורים שלהלן הוא תרשים עמודות של שמות מפעילים. Matplotlib משרטטת סימנים (glyphs) בסדר לוגי (סדר האחסון) ואינה מיישמת את האלגוריתם הדו-כיווני (bidirectional) של Unicode, ולכן טקסט עברי יוצא הפוך ויזואלית. התא שלהלן מטליא (patch) את `matplotlib.text.Text.set_text` פעם אחת, כך שכל מחרוזת עברית מומרת לסדר תצוגה לפני השרטוט. מחרוזות ללא תווים עבריים עוברות ללא שינוי, וההטלאה היא אידמפוטנטית (הרצה חוזרת של התא אינה מזיקה).

In [ ]:
# Stop names are Hebrew. Matplotlib does not apply the Unicode bidi algorithm, so
# Hebrew labels render reversed. Patch it once, before drawing any figure.
_ensure("python-bidi")
import re
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.text as mtext
from bidi.algorithm import get_display

_HEBREW_RE = re.compile(r"[\u0590-\u05FF]")

def fix_he(text):
    """Return display-ordered text. Non-Hebrew is returned untouched."""
    if not isinstance(text, str) or not _HEBREW_RE.search(text):
        return text
    return get_display(text)

def install_hebrew():
    # Arial exists on Windows; DejaVu Sans ships with matplotlib and covers Hebrew.
    matplotlib.rcParams["font.family"] = ["Arial", "DejaVu Sans"]
    matplotlib.rcParams["axes.unicode_minus"] = False
    if getattr(mtext.Text, "_bidi_patched", False):
        return
    _orig = mtext.Text.set_text
    def set_text(self, s):
        if isinstance(s, str) and getattr(self, "_bidi_display", None) == s:
            return _orig(self, s)
        fixed = fix_he(s)
        if isinstance(fixed, str):
            self._bidi_display = fixed
        return _orig(self, fixed)
    mtext.Text.set_text = set_text
    mtext.Text._bidi_patched = True

install_hebrew()

## טעינת טבלאות ה-GTFS הגולמיות

קובצי GTFS הם טקסט פשוט מופרד בפסיקים, אך שלוש החלטות פרסור חשובות דיין כדי לציינן במפורש, משום ששגיאה בכל אחת מהן משבשת את המזהים שכל הגרף מבוסס עליהם:

1. **`dtype=str`** - מזהי GTFS הם מחרוזות אטומות, לא מספרים. אם pandas מסיקה מספרים שלמים, `"01"` ו-`"1"` מתמזגים לאותו ערך והצירופים (joins) בין `trips` ל-`routes` מתחילים להתאים שורות שגויות. אנו שומרים הכול כטקסט וממירים רק את שתי עמודות הקואורדינטות, במכוון, בשלב מאוחר יותר.
2. **`keep_default_na=False`** - שדה GTFS ריק משמעו *לא סופק*, וכמה שמות ותיאורי תחנות אמיתיים בקובץ זה מכילים מחרוזות כגון `NA`. אילו הרשינו ל-pandas להמיר אותם אוטומטית ל-`NaN`, היינו גם ממציאים נתונים חסרים וגם הורסים שמות אמיתיים.
3. **`encoding="utf-8-sig"`** - הקובץ הישראלי מגיע עם סימן סדר בתים (BOM) של UTF-8. בלעדי הגדרה זו, העמודה הראשונה בכל קובץ נקראת עם תו BOM בלתי נראה מודבק לשמה, ו-`df["stop_id"]` זורק `KeyError`.

אנו טוענים את כל ארבע הטבלאות למילון של מסגרות *גולמיות* ומשאירים אותן ללא נגיעה, כך שניתן יהיה להשוות כל ספירת שורות מאוחרת מול המקור.

In [ ]:
GTFS_FILES = {
    "stops":  "stops.txt",
    "routes": "routes.txt",
    "trips":  "trips.txt",
    "agency": "agency.txt",
}

def load_gtfs(name):
    """Load one GTFS table as raw text, with the three parsing decisions above."""
    path = DATA / GTFS_FILES[name]
    if not path.exists():
        raise FileNotFoundError(
            f"{path} is missing. The GTFS feed is tracked in the repository - "
            "check that the clone completed and that DATA points at the feed folder."
        )
    return pd.read_csv(path, dtype=str, keep_default_na=False, encoding="utf-8-sig")

raw = {name: load_gtfs(name) for name in GTFS_FILES}

for name, df in raw.items():
    print(f"{name:7s} rows={len(df):>9,d}  columns={list(df.columns)}")

raw["stops"].head(3)

## ניקוי טבלת התחנות, מסנן אחד בר-ביקורת בכל פעם

טבלת התחנות היא טבלת הצמתים של הגרף העתידי, ולכן היא זוכה לבחינה המדוקדקת ביותר. ארבעה מסננים מוחלים בזה אחר זה, וכל אחד מהם מתעד כמה שורות הסיר לתוך `cleaning_steps`, שמאוחר יותר עובר סריאליזציה לדוח ה-JSON:

1. **קואורדינטות שאינן ניתנות לפרסור.** `stop_lat` / `stop_lon` מומרות עם `errors="coerce"`, כך שמחרוזת ריקה או כל זבל לא-מספרי הופכים ל-`NaN`, והשורות הללו מוסרות. תחנה ללא מיקום אינה יכולה כלל להיות ממוקמת בגרף מרחבי.
2. **קו רוחב בתוך תיבת התיחום** (`29 < lat < 34`). לוכד מצייני מקום של `0.0` ושגיאות הקלדה של החלפת ספרות.
3. **קו אורך בתוך תיבת התיחום** (`34 < lon < 36`). מאותו נימוק.
4. **`stop_id` כפול.** מסנן זה הוא תוספת לסקריפט הניקוי המקורי. `stop_id` הוא מפתח הצומת בכל שלב מאוחר יותר; אילו לא היה ייחודי, פעולת מיזוג הייתה מכפילה שורות ומנפחת את הדרגה (degree) של הצומת הכפול. שמירת המופע הראשון הופכת את ההבטחה למפורשת במקום מונחת מאליה.

**הערה כנה מראש:** בגרסת הקובץ המצורפת למאגר זה, מסננים 1-4 מסירים כולם **אפס** שורות - הקואורדינטות שלמות ובטווח, ו-`stop_id` כבר ייחודי. זו תוצאה טובה, ולא שלב מבוזבז: המספרים המודפסים להלן הם הראיה לטענה, ואותו קוד מגן על צינור העיבוד אם הקובץ ירוענן אי פעם עם ייצוא מלוכלך יותר.

יש לשים לב גם למה שאיננו מסננים במכוון: שורות עם `location_type = 1` הן *תחנות אב* (מכולות הורה) ולא רציפים שניתן לעלות בהם. הן נשארות בטבלה כאן, והדוח סופר אותן, כך שמחברת בניית הגרף תוכל להחליט כיצד לטפל בתחנות אב/בן כשכל המידע מונח לפניה.

In [ ]:
cleaning_steps = []

def record(step, before, after, note):
    """Log one cleaning filter so the report can state exactly what was dropped."""
    cleaning_steps.append({
        "step": step,
        "rows_before": int(before),
        "rows_after": int(after),
        "rows_dropped": int(before - after),
        "note": note,
    })
    print(f"{step:26s} {before:>7,d} -> {after:>7,d}   dropped {before - after:>6,d}   {note}")

stops = raw["stops"].copy()

# 1. Coordinates must be numeric. Empty strings and junk become NaN and are dropped.
stops["stop_lat"] = pd.to_numeric(stops["stop_lat"], errors="coerce")
stops["stop_lon"] = pd.to_numeric(stops["stop_lon"], errors="coerce")
before = len(stops)
stops = stops.dropna(subset=["stop_lat", "stop_lon"])
record("numeric coordinates", before, len(stops), "stop_lat/stop_lon parse as numbers")

# 2. Latitude inside the Israel bounding box.
before = len(stops)
stops = stops[(stops["stop_lat"] > LAT_MIN) & (stops["stop_lat"] < LAT_MAX)]
record("latitude in range", before, len(stops), f"{LAT_MIN} < lat < {LAT_MAX}")

# 3. Longitude inside the Israel bounding box.
before = len(stops)
stops = stops[(stops["stop_lon"] > LON_MIN) & (stops["stop_lon"] < LON_MAX)]
record("longitude in range", before, len(stops), f"{LON_MIN} < lon < {LON_MAX}")

# 4. stop_id is the node key of the graph - it has to be unique.
before = len(stops)
stops = stops.drop_duplicates(subset=["stop_id"], keep="first")
record("unique stop_id", before, len(stops), "stop_id is the graph node key")

stops = stops.reset_index(drop=True)
kept = 100.0 * len(stops) / len(raw["stops"])
print()
print(f"clean stops: {len(stops):,d} of {len(raw['stops']):,d} raw rows kept ({kept:.2f}%)")
print()
print("location_type breakdown of the clean table (0 = stop/platform, 1 = station):")
print(stops["location_type"].value_counts().to_string())

## בדיקת שפיות של הקואורדינטות

לפני שסומכים על מסנן תיבת התיחום, כדאי לבחון את התפרושת בפועל של הנתונים: אם המינימום והמקסימום הנצפים יושבים בנוחות *בתוך* התיבה, המסנן הוא מעקה בטיחות ולא מנגנון שקטע בשקט חלק מהמדינה. הפלט שלהלן מציג את ערכי הקיצון ואת תחנות הפינה, שקל לאמת אותן מול מפה (הקיצון הדרומי אמור להיות אילת, והצפוני - הגליל העליון).

In [ ]:
print(f"latitude  range: {stops['stop_lat'].min():.5f} .. {stops['stop_lat'].max():.5f}   (box {LAT_MIN} .. {LAT_MAX})")
print(f"longitude range: {stops['stop_lon'].min():.5f} .. {stops['stop_lon'].max():.5f}   (box {LON_MIN} .. {LON_MAX})")
print()

extremes = pd.DataFrame([
    stops.loc[stops["stop_lat"].idxmin()],
    stops.loc[stops["stop_lat"].idxmax()],
    stops.loc[stops["stop_lon"].idxmin()],
    stops.loc[stops["stop_lon"].idxmax()],
])
extremes.index = ["southernmost", "northernmost", "westernmost", "easternmost"]
extremes[["stop_id", "stop_name", "stop_lat", "stop_lon"]]

## תוויות אזור ומטרופולין

שלבים מאוחרים יותר משווים מרכזיות ועמידות בין חלקי המדינה, ולכן כל תחנה זקוקה לתווית גאוגרפית. הקובץ אינו מספק אף תווית - ל-`stops.txt` אין עמודת מחוז מנהלי - ולכן אנו גוזרים שתי תוויות מהקואורדינטות בלבד.

**`region`** (הכלל המתאים הראשון מנצח, בשחזור מדויק של שרשרת ה-`if/elif` המקורית):

| סדר | כלל | תווית |
|---|---|---|
| 1 | `31.70 <= lat <= 31.90` **וגם** `34.95 <= lon <= 35.30` | Jerusalem |
| 2 | `lat > 32.50` | North |
| 3 | `lat >= 31.55` | Center |
| 4 | אחרת | South |

**`metro`** - המרכז הראשון שהדיסקה שלו מכילה את התחנה: Tel Aviv (30 ק"מ), Haifa (25 ק"מ), Jerusalem (20 ק"מ), Beer Sheva (25 ק"מ); כל היתר מסווג כ-`Periphery`. המרחקים הם מרחקי מעגל גדול (haversine).

### מגבלות - נא לקרוא זאת לפני שימוש בתוויות

**הקצאת ה-region היא כלל גס של קווי רוחב/אורך, ולא גאוגרפיה מנהלית או סטטיסטית אמיתית.** היא אינה יודעת דבר על גבולות מחוזות, גבולות מוניציפליים, הקו הירוק, טופוגרפיה או זמן נסיעה. באופן קונקרטי:

- אזורים 2/3/4 הם חתכי קו רוחב טהורים, ולכן הם רצועות אופקיות לרוחב המדינה. תחנה בשפלת הנגב המערבי ותחנה במדבר יהודה המזרחי באותו קו רוחב מקבלות את אותה תווית, אף שהן שייכות למציאויות תחבורה שונות לחלוטין.
- אזור Jerusalem הוא מלבן, ולכן תחנות בפרוזדור ירושלים ממש מחוץ לתיבה נופלות תחת "Center", והמלבן נבדק *ראשון*, כלומר הוא גובר על רצועות קווי הרוחב בכל מקום שבו הם חלוקים.
- דיסקות ה-metro הן מעגלים סביב נקודת מרכז עיר יחידה עם רדיוסים שנבחרו ידנית. דיסקה של 30 ק"מ סביב תל אביב חורגת הרבה מעבר לגוש דן הסטטיסטי בכיוונים מסוימים ואינה מגיעה אליו בכיוונים אחרים.
- `region` ו-`metro` מחושבים באופן בלתי תלוי ועשויים לסתור זה את זה - תחנה יכולה להיות `region = Center` וגם `metro = Jerusalem`.

לפיכך התוויות מתאימות לקיבוץ תיאורי גס ("בערך כמה מהרשת יושבת בצפון?") ו**אין** להציגן כסטטיסטיקה אזורית רשמית. הסקריפט המקורי השתמש בתוויות בעברית; הן תורגמו כאן לאנגלית (North / Center / Jerusalem / South, ו-Periphery עבור מה שאינו מטרופוליני) ללא שינוי בערכי הסף.

הערת מימוש אחת: הקוד המקורי קרא לפונקציית haversine סקלרית בתוך `DataFrame.apply` פעם אחת לכל תחנה, כלומר כ-35,000 קריאות פונקציה ברמת Python. הגרסה שלהלן מחשבת את אותם מספרים בצורה וקטורית באמצעות NumPy - תוצאות זהות, בשבריר מזמן הריצה. התא גם מדפיס את המרחקים בין כל זוג מבין ארבעת מרכזי המטרופולין, כדי לוודא שהדיסקות אינן חופפות; אם אין חפיפה, אזי "ההתאמה הראשונה מנצחת" ו"המרכז הקרוב ביותר מנצח" הם אותו כלל, וסדר המילון אינו נושא הטיה סמויה.

In [ ]:
# Metro centres: name -> (latitude, longitude, radius in km)
METRO_CENTERS = {
    "Tel Aviv":   (32.0853, 34.7818, 30),
    "Haifa":      (32.7940, 34.9896, 25),
    "Jerusalem":  (31.7683, 35.2137, 20),
    "Beer Sheva": (31.2518, 34.7913, 25),
}

def haversine_km(lat1, lon1, lat2, lon2):
    """Great-circle distance in km. Accepts scalars or NumPy arrays."""
    R = 6371.0
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(np.asarray(lat2) - np.asarray(lat1))
    dlam = np.radians(np.asarray(lon2) - np.asarray(lon1))
    a = np.sin(dphi / 2.0) ** 2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlam / 2.0) ** 2
    return 2.0 * R * np.arcsin(np.sqrt(a))

lat = stops["stop_lat"].to_numpy()
lon = stops["stop_lon"].to_numpy()

# --- Region: np.select evaluates conditions in order, exactly like if/elif/else ---
region_conditions = [
    (lat >= 31.70) & (lat <= 31.90) & (lon >= 34.95) & (lon <= 35.30),  # Jerusalem box
    lat > 32.50,                                                        # North
    lat >= 31.55,                                                       # Center
]
stops["region"] = np.select(region_conditions, ["Jerusalem", "North", "Center"], default="South")

# --- Metro: first centre whose disc contains the stop ---
metro_conditions, metro_names = [], []
for city, (clat, clon, radius) in METRO_CENTERS.items():
    metro_conditions.append(haversine_km(lat, lon, clat, clon) <= radius)
    metro_names.append(city)
stops["metro"] = np.select(metro_conditions, metro_names, default="Periphery")

# --- Do the metro discs overlap? If not, dictionary order is irrelevant. ---
print("metro disc overlap check")
items = list(METRO_CENTERS.items())
for i in range(len(items)):
    for j in range(i + 1, len(items)):
        a_name, (a_lat, a_lon, a_r) = items[i]
        b_name, (b_lat, b_lon, b_r) = items[j]
        d = float(haversine_km(a_lat, a_lon, b_lat, b_lon))
        verdict = "OVERLAP - order matters" if d < a_r + b_r else "disjoint"
        print(f"  {a_name:11s} - {b_name:11s}: {d:6.1f} km apart, radii sum {a_r + b_r:3d} km -> {verdict}")

print()
print("stops per region")
print(stops["region"].value_counts().to_string())
print()
print("stops per metro")
print(stops["metro"].value_counts().to_string())
print()
print("region x metro cross-tabulation (they are computed independently and can disagree)")
pd.crosstab(stops["region"], stops["metro"])

## קווים, נסיעות, מפעילים ושלמות רפרנציאלית

שלוש הטבלאות האחרות אינן זקוקות לסינון שורות - הן טבלאות מזהים ולא מדידות - אך הן כן זקוקות ל*תיאור*, ויש לבדוק את שלמותן הרפרנציאלית לפני שמחברות מאוחרות יותר מצרפות אותן זו לזו.

שני דברים מחושבים כאן:

1. **הרכב אופני התחבורה.** `route_type` הוא קוד אופן התחבורה של GTFS (0 חשמלית/רכבת קלה, 2 רכבת, 3 אוטובוס, וכן הלאה). ספירת *קווים* לכל אופן תחבורה מלמדת כיצד לוח הזמנים מאורגן; ספירת *נסיעות מתוזמנות* לכל אופן תחבורה מלמדת כמה שירות כל אופן מפעיל בפועל, וזהו המספר הרלוונטי בשקלול הרשת. שניהם מדווחים.
2. **בדיקות שלמות.** כל `trip` שה-`route_id` שלו נעדר מ-`routes.txt`, או כל `route` שה-`agency_id` שלו נעדר מ-`agency.txt`, היה הופך בשקט ל-`NaN` לאחר צירוף שמאלי (left join) ומעוות בחשאי צבירים לפי אופן תחבורה או לפי מפעיל. אנו סופרים אותם כאן ומכניסים את הספירות לדוח; ערך שאינו אפס הוא סימן אזהרה עבור כל שלב במורד הזרם.

In [ ]:
# Human-readable names for the GTFS route_type codes present in this feed.
ROUTE_TYPE_LABELS = {
    "0": "tram/light rail",
    "1": "subway",
    "2": "rail",
    "3": "bus",
    "4": "ferry",
    "5": "cable tram",
    "6": "aerial lift",
    "7": "funicular",
    "8": "trolleybus",
    "715": "demand/other bus",
}

routes = raw["routes"].copy()
trips = raw["trips"].copy()
agencies = raw["agency"].copy()

# Routes per mode.
route_types = (routes["route_type"]
               .value_counts()
               .rename_axis("route_type")
               .reset_index(name="routes"))
route_types["route_type_label"] = route_types["route_type"].map(ROUTE_TYPE_LABELS).fillna("unknown")

# Scheduled trips per mode - the service-volume view of the same breakdown.
trips_by_type = (trips.merge(routes[["route_id", "route_type"]], on="route_id", how="left")
                      .groupby("route_type")
                      .size()
                      .rename("scheduled_trips")
                      .reset_index())
route_types = route_types.merge(trips_by_type, on="route_type", how="left")
route_types["scheduled_trips"] = route_types["scheduled_trips"].fillna(0).astype(int)
route_types = route_types[["route_type", "route_type_label", "routes", "scheduled_trips"]]

# Referential integrity: dangling foreign keys would become silent NaNs in later joins.
orphan_trips = int((~trips["route_id"].isin(set(routes["route_id"]))).sum())
orphan_routes = int((~routes["agency_id"].isin(set(agencies["agency_id"]))).sum())
print("trips whose route_id is not in routes.txt  :", orphan_trips)
print("routes whose agency_id is not in agency.txt:", orphan_routes)
print()

# Scheduled trips per operator.
agency_summary = (trips.merge(routes[["route_id", "agency_id"]], on="route_id", how="left")
                       .merge(agencies[["agency_id", "agency_name"]], on="agency_id", how="left")
                       .groupby(["agency_id", "agency_name"])
                       .size()
                       .rename("scheduled_trips")
                       .reset_index()
                       .sort_values("scheduled_trips", ascending=False)
                       .reset_index(drop=True))

print(f"agencies: {len(agencies):,d}   routes: {len(routes):,d}   trips: {len(trips):,d}")
route_types

## איורים תיאוריים

ארבעה איורים, שכל אחד מהם עונה על שאלה אחת בנוגע לנתונים המנוקים, וכולם נשמרים כקובצי PNG לתוך תיקיית `figures/` של השלב:

1. **תחנות לפי region ולפי metro** - האם קבוצת הצמתים מאוזנת, או שמא היא נשלטת על ידי חלק אחד של המדינה?
2. **פיזור גאוגרפי של כל תחנה מנוקה, בצביעה לפי region** - בדיקת השפיות החשובה ביותר. אילו מסנן תיבת התיחום או כלל ה-region היו שבורים, הדבר היה נראה כאן מיד כנקודות תועות בים או כרצועה במקום הלא נכון. יחס הממדים מתוקן באמצעות `cos(latitude)` כדי שהמדינה לא תימתח אופקית.
3. **קווים ונסיעות מתוזמנות לפי אופן תחבורה** - על ציר לוגריתמי, משום ששירות האוטובוסים שולט בקובץ בסדרי גודל.
4. **המפעילים המובילים לפי נסיעות מתוזמנות** - שמות מפעילים בעברית, המוצגים דרך הטלאי הדו-כיווני (bidi) שהותקן קודם לכן.

In [ ]:
REGION_COLORS = {"North": "#1d4ed8", "Center": "#0f766e", "Jerusalem": "#b45309", "South": "#be123c"}

# --- Figure 1: stops per region and per metro ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
region_counts = stops["region"].value_counts()
axes[0].bar(region_counts.index, region_counts.values,
            color=[REGION_COLORS.get(r, "#64748b") for r in region_counts.index])
axes[0].set_title("Stops per region (crude lat/lon rule)")
axes[0].set_ylabel("number of stops")
metro_counts = stops["metro"].value_counts()
axes[1].bar(metro_counts.index, metro_counts.values, color="#334155")
axes[1].set_title("Stops per metropolitan area")
axes[1].tick_params(axis="x", rotation=20)
for ax in axes:
    ax.grid(axis="y", alpha=0.3)
    ax.set_axisbelow(True)
fig.tight_layout()
fig.savefig(FIGURES / "region_and_metro_counts.png", dpi=150)
plt.show()

### איור 2 - המפה

זוהי בדיקת השפיות היחידה החזקה ביותר במחברת. שרטוט כל התחנות המנוקות במרחב קו אורך/קו רוחב אמור לשחזר את צורתה המזוהה של המדינה: רצועת חוף צפופה, פרוזדור ירושלים, ופיזור דליל היורד לאורך הנגב עד אילת. כל דבר ששרד את מסנן תיבת התיחום ובכל זאת שגוי - קואורדינטה בים, קו רוחב משוקף - מופיע כאן כחריג בולט. הצביעה לפי `region` ממחישה בו-זמנית את חתכי קווי הרוחב הגסים, כך שהקורא יוכל לראות בעצמו היכן הכלל שרירותי. מרכזי המטרופולין מסומנים כדי שניתן יהיה להעריך את רדיוסי הדיסקות בעין.

In [ ]:
# --- Figure 2: every clean stop on the map, coloured by region ---
fig, ax = plt.subplots(figsize=(6.5, 9))
for name, part in stops.groupby("region"):
    ax.scatter(part["stop_lon"], part["stop_lat"], s=1.5, alpha=0.35,
               color=REGION_COLORS.get(name, "#64748b"),
               label=f"{name} (n={len(part):,d})")
for city, (clat, clon, radius) in METRO_CENTERS.items():
    ax.plot(clon, clat, marker="o", markersize=7, color="black", zorder=5)
    ax.annotate(city, (clon, clat), textcoords="offset points", xytext=(8, 4), fontsize=9, zorder=6)
ax.set_aspect(1.0 / math.cos(math.radians(31.7)))  # keep the country from looking stretched
ax.set_xlabel("longitude")
ax.set_ylabel("latitude")
ax.set_title(f"{len(stops):,d} clean stops, coloured by assigned region")
ax.legend(markerscale=8, loc="upper left", fontsize=9)
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(FIGURES / "stops_map_by_region.png", dpi=150)
plt.show()

### איור 3 - הרכב השירות לפי אופן תחבורה

קווים ונסיעות מתוזמנות זה לצד זה, לפי `route_type` של GTFS. ציר ה-y לוגריתמי משום ששירות האוטובוסים גדול בכמה סדרי גודל מכל אופן תחבורה אחר; על ציר ליניארי עמודות הרכבת והרכבת הקלה היו בלתי נראות. הפער בין שתי העמודות של אותו אופן תחבורה הוא מידע בפני עצמו: אופן תחבורה עם מעט קווים אך נסיעות רבות מפעיל שירות בתדירות גבוהה על רשת קטנה.

In [ ]:
# --- Figure 3: routes and scheduled trips per GTFS mode (log scale) ---
labels = route_types["route_type_label"] + " (" + route_types["route_type"] + ")"
x = np.arange(len(route_types))
fig, ax = plt.subplots(figsize=(9, 4.8))
ax.bar(x - 0.2, route_types["routes"], width=0.4, label="routes", color="#0f766e")
ax.bar(x + 0.2, route_types["scheduled_trips"], width=0.4, label="scheduled trips", color="#b45309")
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=15)
ax.set_yscale("log")
ax.set_ylabel("count (log scale)")
ax.set_title("Service composition by GTFS mode")
ax.legend()
ax.grid(axis="y", alpha=0.3)
ax.set_axisbelow(True)
fig.tight_layout()
fig.savefig(FIGURES / "route_type_distribution.png", dpi=150)
plt.show()

### איור 4 - מפעילים

נסיעות מתוזמנות לכל מפעיל, 12 המובילים. זוהי בדיקת ריכוזיות: התחבורה הציבורית בישראל מופעלת על ידי מפעילים מורשים רבים, אך צפוי שנפח השירות יהיה נשלט על ידי קומץ מהם. שמות המפעילים הם בעברית ומוצגים דרך הטלאי הדו-כיווני (bidi) שהותקן קודם לכן - אם באיור זה מופיע טקסט הפוך, סימן שהטלאי לא נכנס לתוקף.

In [ ]:
# --- Figure 4: top operators by scheduled trips (Hebrew names via the bidi patch) ---
top_agencies = agency_summary.head(12).iloc[::-1]
fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(top_agencies["agency_name"], top_agencies["scheduled_trips"], color="#0f766e")
ax.set_xlabel("scheduled trips")
ax.set_title("Top 12 operators by number of scheduled trips")
ax.grid(axis="x", alpha=0.3)
ax.set_axisbelow(True)
fig.tight_layout()
fig.savefig(FIGURES / "top_agencies_by_trips.png", dpi=150)
plt.show()

## שמירת תוצרי השלב

כל מה שהמחברות המאוחרות זקוקות לו נכתב לתוך `outputs/nb/01_data_preparation/tables/`. ארבעת קובצי `*_clean.csv` נכתבים עם `utf-8-sig` כדי שהשמות בעברית ייפתחו כראוי ב-Excel, ועם `index=False` כדי שקריאה חוזרת שלהם לא תוסיף עמודת רפאים.

דוח ה-JSON הוא התוצר שהופך את השלב הזה לבר-ביקורת: הוא שומר את ספירות השורות לכל מסנן, את תיבת התיחום שנעשה בה שימוש בפועל, את הגדרות ה-region וה-metro, את הפילוחים לפי אופן תחבורה ולפי סוג מיקום, ואת ספירות השלמות הרפרנציאלית. בודק יכול לקרוא את הדוח בלבד ולדעת בדיוק מה הוסר ומדוע, בלי להריץ מחדש את המחברת.

נקודה טכנית קטנה אחת: `Series.value_counts().to_dict()` עשוי להחזיר ערכים מטיפוס integer של NumPy, ש-`json.dump` מסרב לבצע להם סריאליזציה בחלק מהשילובים של pandas/NumPy. פונקציית העזר שלהלן ממירה מפתחות ל-`str` וערכים ל-`int` רגיל, כך שהדוח תמיד עובר סריאליזציה.

כתיבת `trips_clean.csv` (כ-28 MB) היא החלק האיטי; יש להגדיר `SAVE_TRIPS_CSV = False` בתא הקבועים כדי לדלג עליה.

In [ ]:
def counts_to_dict(series):
    """value_counts as a JSON-safe {str: int} dictionary."""
    return {str(k): int(v) for k, v in series.value_counts().items()}

# --- cleaned tables ---
stops.to_csv(TABLES / "stops_clean.csv", index=False, encoding="utf-8-sig")
routes.to_csv(TABLES / "routes_clean.csv", index=False, encoding="utf-8-sig")
agencies.to_csv(TABLES / "agencies_clean.csv", index=False, encoding="utf-8-sig")
if SAVE_TRIPS_CSV:
    trips.to_csv(TABLES / "trips_clean.csv", index=False, encoding="utf-8-sig")
else:
    print("skipped trips_clean.csv (SAVE_TRIPS_CSV = False)")

# --- derived summary tables ---
route_types.to_csv(TABLES / "route_type_distribution.csv", index=False, encoding="utf-8-sig")
agency_summary.to_csv(TABLES / "agency_trip_counts.csv", index=False, encoding="utf-8-sig")

# --- the audit trail ---
report = {
    "generated_by": "notebooks/01_data_preparation.ipynb",
    "gtfs_source_dir": str(DATA),
    "stops_raw": int(len(raw["stops"])),
    "stops_clean": int(len(stops)),
    "stops_dropped": int(len(raw["stops"]) - len(stops)),
    "routes_total": int(len(routes)),
    "trips_total": int(len(trips)),
    "agencies_total": int(len(agencies)),
    "cleaning_steps": cleaning_steps,
    "coordinate_bounds": {
        "lat_min": LAT_MIN, "lat_max": LAT_MAX,
        "lon_min": LON_MIN, "lon_max": LON_MAX,
    },
    "observed_bounds": {
        "lat_min": float(stops["stop_lat"].min()), "lat_max": float(stops["stop_lat"].max()),
        "lon_min": float(stops["stop_lon"].min()), "lon_max": float(stops["stop_lon"].max()),
    },
    "region_rule": (
        "crude first-match lat/lon rule: Jerusalem box (31.70-31.90 lat, 34.95-35.30 lon); "
        "North lat > 32.50; Center lat >= 31.55; South otherwise. Not an administrative geography."
    ),
    "metro_centers": {
        city: {"lat": c[0], "lon": c[1], "radius_km": c[2]} for city, c in METRO_CENTERS.items()
    },
    "region_counts": counts_to_dict(stops["region"]),
    "metro_counts": counts_to_dict(stops["metro"]),
    "location_type_counts": counts_to_dict(stops["location_type"]),
    "route_type_counts": counts_to_dict(routes["route_type"]),
    "integrity": {
        "trips_with_unknown_route_id": orphan_trips,
        "routes_with_unknown_agency_id": orphan_routes,
    },
}

report_path = TABLES / "data_cleaning_report.json"
with open(report_path, "w", encoding="utf-8") as fh:
    json.dump(report, fh, ensure_ascii=False, indent=2)

print("written to", TABLES)
for p in sorted(TABLES.iterdir()):
    print(f"  {p.name:32s} {p.stat().st_size / 1024**2:8.2f} MB")
print()
print(json.dumps({k: v for k, v in report.items() if k != "cleaning_steps"},
                 ensure_ascii=False, indent=2)[:2000])

## מסקנות

- **הקובץ נקי.** כל ארבעת מסנני הניקוי - קואורדינטות שאינן ניתנות לפרסור, טווח קו רוחב, טווח קו אורך, ו-`stop_id` כפול - מסירים **אפס** שורות מהגרסה המצורפת של הקובץ: לכל תחנה יש קואורדינטות מספריות, כל קואורדינטה נופלת בתוך תיבת התיחום של ישראל (התפרושת הנצפית היא בקירוב 29.49-33.28 צפון, 34.28-35.84 מזרח, בנוחות בתוך תיבת המגן 29-34 / 34-36), ו-`stop_id` כבר ייחודי. זוהי תוצאה שלילית עבור שלב הניקוי ותוצאה חיובית עבור הנתונים: קבוצת הצמתים של הרשת היא טבלת התחנות הגולמית במלואה. המסננים נותרים בצינור העיבוד כמעקי בטיחות לרענוני קובץ עתידיים, והדוח מוכיח שהם נבדקו ולא הונחו מאליהם.
- **טבלת התחנות גדולה ומוטה מאוד לאוטובוסים.** כ-35,000 תחנות, שמתוכן רק כמה מאות הן תחנות אב עם `location_type = 1`; `route_type = 3` (אוטובוס) שולט בסדרי גודל הן במספר הקווים והן במספר הנסיעות המתוזמנות. כל מסקנה על "רשת התחבורה הציבורית בישראל" הנגזרת מהגרף המלא היא למעשה מסקנה על רשת האוטובוסים, ומשום כך הפרויקט מנתח גם את שכבת הרכבת בנפרד.
- **תוויות region ו-metro הן תוויות נוחות, לא גאוגרפיה.** ה-region מורכב משלושה חתכי קו רוחב בתוספת מלבן אחד סביב ירושלים; ה-metro מורכב מארבעה מעגלים עם רדיוסים שנבחרו ידנית. הן שימושיות לקיבוץ גס ולצביעת מפות, ואין לצטט אותן כסטטיסטיקה אזורית. שתי התוויות מחושבות באופן בלתי תלוי, וטבלת ההצלבה שהודפסה למעלה מראה היכן הן חלוקות. הביטחון המבני היחיד הוא שארבע דיסקות ה-metro זרות זו לזו בכל זוג, ולכן "ההתאמה הראשונה מנצחת" שקול ל"המרכז הקרוב ביותר מנצח" והסדר אינו מכניס הטיה.
- **שלמות רפרנציאלית אומתה, ולא הונחה מאליה.** ספירות הנסיעות עם `route_id` לא מוכר והקווים עם `agency_id` לא מוכר מחושבות ונשמרות בדוח; מחברות מאוחרות יותר מצרפות את הטבלאות הללו בחופשיות ומסתמכות על כך שספירות אלו הן אפס.
- **מה שהושאר במכוון לשלב 02:** עדיין לא קיימות קשתות. שום דבר כאן אינו קורא את `stop_times.txt`, ולכן טרם הוצג כל מושג של שכנות (adjacency), מרווח שירות (headway) או זמן נסיעה. גם איחוד תחנות אב/בן הושאר ללא נגיעה במתכוון, כדי שמחברת בניית הגרף תוכל לקבל את ההחלטה הזו כשטבלת התחנות המלאה מונחת לפניה.